# 05 — Production chronological split and baseline forecasts

**Purpose:** use the same calendar-based train/validation/test split as the released production training pipeline and evaluate simple baselines.

This notebook imports `load_training_data()` and `chronological_split()` directly from `src/stockout_radar/training.py`. This avoids maintaining a second, potentially different split implementation.

The target covers the forecast-origin week plus the next three weeks. The production helper reserves a four-week separation between the train, validation and test periods to reduce overlap between target windows. Never randomly split individual product-week rows for this task.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from stockout_radar.training import load_training_data, chronological_split

model_df = load_training_data()
train, val, test, split_info = chronological_split(model_df)

for name, frame in [("train", train), ("validation", val), ("test", test)]:
    print(
        name,
        frame.shape,
        frame["week"].min().date(),
        "to",
        frame["week"].max().date(),
    )

print("\nProduction split metadata:")
print(split_info)

train (236926, 19) 2010-02-01 to 2011-04-25
validation (46123, 19) 2011-05-23 to 2011-08-01
test (48511, 19) 2011-08-29 to 2011-11-07

Production split metadata:
{'train_start': '2010-02-01', 'train_end': '2011-04-25', 'validation_start': '2011-05-23', 'validation_end': '2011-08-01', 'test_start': '2011-08-29', 'test_end': '2011-11-07', 'embargo_weeks': 4}


## Evaluation metrics

- **MAE:** average absolute error in units.
- **RMSE:** square root of average squared error; larger errors receive more weight.
- **WAPE:** sum of absolute errors divided by sum of absolute actual demand, expressed as a percentage.

WAPE is an aggregate measure and can be dominated by high-volume products. It should be read alongside MAE, RMSE and product-level diagnostics.

In [2]:
def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    error = y_pred - y_true

    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error ** 2))
    denominator = np.sum(np.abs(y_true))
    wape_percent = (
        np.sum(np.abs(error)) / denominator * 100
        if denominator > 0 else np.nan
    )

    return {
        "MAE (units)": round(float(mae), 4),
        "RMSE (units)": round(float(rmse), 4),
        "WAPE (%)": round(float(wape_percent), 4),
    }

## Baseline comparison on validation data

The naive baseline repeats the previous week's units four times. The moving-average baseline multiplies the mean of the previous four weeks by four. These are simple reference forecasts, not learned models.

In [3]:
validation_results = {
    "Naive (4 x lag1)": evaluate(
        val["target"], 4 * val["units_lag1"]
    ),
    "Moving average (4 x mean4)": evaluate(
        val["target"], 4 * val["units_mean4"]
    ),
}

pd.DataFrame(validation_results).T

,MAE (units),RMSE (units),WAPE (%)
Naive (4 x lag1),60.9029,257.1572,78.5286
Moving average (4 x mean4),45.3431,162.6715,58.4657


## Baseline comparison on the untouched test period

The test period is kept separate from model selection. These baseline results provide the reference values that a selected model should be compared against on exactly the same test rows.

In [4]:
test_results = {
    "Naive (4 x lag1)": evaluate(
        test["target"], 4 * test["units_lag1"]
    ),
    "Moving average (4 x mean4)": evaluate(
        test["target"], 4 * test["units_mean4"]
    ),
}

pd.DataFrame(test_results).T

,MAE (units),RMSE (units),WAPE (%)
Naive (4 x lag1),83.9026,326.1412,67.6253
Moving average (4 x mean4),66.1085,236.2272,53.2833


## Connection to the final production report

The authoritative model-selection and test results are stored in `reports/metrics/training_evaluation.json` and the diagnostic report in `reports/metrics/holdout_evaluation.json`. The production model is trained by `src/stockout_radar/training.py`; this notebook only establishes the official split and benchmark forecasts.